<div align="center">

<img width="80%" src="https://user-images.githubusercontent.com/73097560/115834477-dbab4500-a447-11eb-908a-139a6edaec5c.gif"/>

<img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg"
     alt="Python" width="80" height="80"/>

<h1 align="center">Bibliotecas Fundamentais de Python</h1>

<h3 align="center">PhD. Julles Mitoura</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3.11-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
  <img src="https://img.shields.io/badge/NumPy-2.x-013243?style=for-the-badge&logo=numpy&logoColor=white"/>
</div>

<img width="80%" src="https://user-images.githubusercontent.com/73097560/115834477-dbab4500-a447-11eb-908a-139a6edaec5c.gif"/>

<br>

<h1 style="font-size:2em; margin: 4px 0;">Regressão Linear com NumPy</h1>

<p style="font-size:13px; opacity:0.6; margin-top:6px;"><em>Estimando a degradação de eficiência de um trocador de calor</em></p>

<br>

</div>

---

## O que é NumPy?

**NumPy** (*Numerical Python*) é a biblioteca fundamental para computação científica em Python. Criada em 2005 por Travis Oliphant, fornece o tipo `ndarray` — um array multidimensional armazenado em memória contígua (C/Fortran) — e uma vasta coleção de funções matemáticas vetorizadas.

### Por que NumPy é relevante?

**Performance**: operações vetorizadas em C, tipicamente **10–100× mais rápidas** que laços Python puros.

| Operação (1 M elementos) | Python puro | NumPy |
|---|---|---|
| Soma | ~80 ms | ~1 ms |
| Produto elemento a elemento | ~120 ms | ~2 ms |
| Norma euclidiana | ~200 ms | ~3 ms |

**Fundação de toda a stack científica** — Pandas, Matplotlib, Seaborn, Scikit-learn, PyTorch e TensorFlow são todos construídos sobre `ndarray`.

### O objeto central: `ndarray`

```
ndarray
├── data     — buffer contíguo em C (float64, int32, ...)
├── shape    — dimensões (linhas, colunas, ...)
├── dtype    — tipo de dado de cada elemento
└── strides  — bytes entre elementos consecutivos em cada eixo
```

### Operações fundamentais com NumPy

| Função | Uso |
|---|---|
| `np.array`, `np.zeros`, `np.linspace` | criação de arrays |
| `np.mean`, `np.std`, `np.sum` | estatísticas descritivas |
| `np.polyfit`, `np.polyval` | regressão polinomial |
| `np.linalg.solve`, `np.linalg.eig` | álgebra linear |

> Nesta disciplina usamos NumPy para implementar a regressão linear (OLS) **do zero** e calcular métricas de avaliação (MSE, RMSE, R²) sobre os dados do trocador de calor.

---

---
## <span style="color:#1E90FF;">1. Importação</span>

In [1]:
import numpy as np
import pandas as pd

---
## <span style="color:#1E90FF;">2. Introdução ao NumPy</span>

In [2]:
# array unidimensional
a = np.array([1, 2, 3, 4, 5])
print('Array:', a)
print('Shape:', a.shape)
print('Dtype:', a.dtype)

Array: [1 2 3 4 5]
Shape: (5,)
Dtype: int64


In [3]:
# operações vetorizadas — aplicadas a todos os elementos sem loop
print('Dobro:    ', a * 2)
print('Quadrado: ', a ** 2)
print('Raiz:     ', np.sqrt(a).round(3))

Dobro:     [ 2  4  6  8 10]
Quadrado:  [ 1  4  9 16 25]
Raiz:      [1.    1.414 1.732 2.    2.236]


In [4]:
# funções estatísticas básicas
print('Média:         ', np.mean(a))
print('Desvio padrão: ', np.std(a).round(4))
print('Mínimo / Máx:  ', np.min(a), '/', np.max(a))
print('Soma:          ', np.sum(a))

Média:          3.0
Desvio padrão:  1.4142
Mínimo / Máx:   1 / 5
Soma:           15


In [5]:
# arange e linspace — gerar sequências numéricas
print('arange(0, 10, 2): ', np.arange(0, 10, 2))   # passo fixo
print('linspace(0, 1, 6):', np.linspace(0, 1, 6))  # n pontos igualmente espaçados

arange(0, 10, 2):  [0 2 4 6 8]
linspace(0, 1, 6): [0.  0.2 0.4 0.6 0.8 1. ]


---
## <span style="color:#1E90FF;">3. Carregando e Preparando os Dados</span>

In [6]:
df = pd.read_csv('data/heat_exchanger.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

numericas = df.select_dtypes(include='number').columns
df[numericas] = df[numericas].interpolate(method='linear')

df.head()

,timestamp,t_water_in,t_glycol_int,t_glycol_out,t_water_out,efficiency
0,2025-09-29,33.49,85.82,46.50,61.808856,96.454517
1,2025-09-30,33.51,86.35,46.75,61.894941,96.432576
2,2025-10-01,33.39,85.98,46.55,61.672872,96.410674
3,2025-10-02,33.32,85.71,46.51,61.403079,96.388810
4,2025-10-03,32.86,87.54,46.71,61.583481,96.366986


In [7]:
# criar variável de dias desde o início da operação
df['dia'] = (df['timestamp'] - df['timestamp'].min()).dt.days

print('Período de operação: %d dias' % df['dia'].max())
print('Datas: %s → %s' % (df['timestamp'].min().date(), df['timestamp'].max().date()))
df[['timestamp', 'dia', 'efficiency']].head()

Período de operação: 180 dias
Datas: 2025-09-29 → 2026-03-28


,timestamp,dia,efficiency
0,2025-09-29,0,96.454517
1,2025-09-30,1,96.432576
2,2025-10-01,2,96.410674
3,2025-10-02,3,96.388810
4,2025-10-03,4,96.366986


In [8]:
# extrair arrays NumPy
x = df['dia'].values        # variável independente: dias
y = df['efficiency'].values # variável dependente: eficiência

print('x (dias):       tipo=%s  shape=%s  min=%d  max=%d' % (type(x).__name__, x.shape, x.min(), x.max()))
print('y (eficiência): tipo=%s  shape=%s  min=%.4f  max=%.4f' % (type(y).__name__, y.shape, y.min(), y.max()))

x (dias):       tipo=ndarray  shape=(175,)  min=0  max=180
y (eficiência): tipo=ndarray  shape=(175,)  min=93.2253  max=96.4545


---
## <span style="color:#1E90FF;">4. Regressão Linear com np.polyfit</span>

A regressão linear ajusta uma reta **`y = ax + b`** que minimiza o erro quadrático entre os valores reais e os estimados.

- **`np.polyfit(x, y, 1)`** → encontra os coeficientes `[a, b]`
- **`np.polyval(coef, x)`** → aplica o polinômio para gerar predições

In [9]:
coef = np.polyfit(x, y, deg=1)
a, b = coef

print('Equação do modelo:')
print(f'  eficiência = {a:.6f} × dia + {b:.6f}')
print()
print(f'Coeficiente angular (a): {a:.6f}  → queda de {abs(a)*30:.4f}% por mês')
print(f'Intercepto (b):          {b:.6f}  → eficiência estimada no dia 0')

Equação do modelo:
  eficiência = -0.017919 × dia + 96.342827

Coeficiente angular (a): -0.017919  → queda de 0.5376% por mês
Intercepto (b):          96.342827  → eficiência estimada no dia 0


Em geral, o modelo que desenvolvemos é o seguinte:

$$Eff = -0.017919.dia + 96.342827$$

In [10]:
# gerar predições e comparar com os valores reais
y_pred = np.polyval(coef, x)

comparacao = pd.DataFrame({
    'dia':                x,
    'eficiência real':    y,
    'eficiência predita': y_pred.round(4),
    'erro':             (y - y_pred).round(3)
}).head()
comparacao

,dia,eficiência real,eficiência predita,erro
0,0,96.454517,96.3428,0.112
1,1,96.432576,96.3249,0.108
2,2,96.410674,96.3070,0.104
3,3,96.388810,96.2891,0.100
4,4,96.366986,96.2712,0.096


---
## <span style="color:#1E90FF;">5. Avaliação do Modelo</span>

**Erro Quadrático Médio (MSE)** — penaliza erros grandes ao elevar ao quadrado:

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

**Raiz do Erro Quadrático Médio (RMSE)** — mesma unidade da variável original:

$$RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

**Coeficiente de Determinação (R²)** — proporção da variância explicada pelo modelo (0 a 1):

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\displaystyle\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\displaystyle\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

onde $\hat{y}_i$ é o valor predito e $\bar{y}$ é a média dos valores reais.

In [11]:
residuos = y - y_pred

mse  = np.mean(residuos ** 2)
rmse = np.sqrt(mse)

ss_res = np.sum(residuos ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
r2 = 1 - ss_res / ss_tot

print(f'MSE:  {mse:.6f}')
print(f'RMSE: {rmse:.6f}  (unidade: % de eficiência)')
print(f'R²:   {r2:.4f}  ({r2*100:.2f}% da variância explicada pelo modelo)')

MSE:  0.002199
RMSE: 0.046892  (unidade: % de eficiência)
R²:   0.9975  (99.75% da variância explicada pelo modelo)


---